# 02 Preprocessing Pipeline

## MobileSkinXAI Thesis Project

This notebook prepares the datasets for model development.

The main goal is to create a clean and reproducible 6-class cross-dataset pipeline using:

- **ISIC 2019** as the primary dataset
- **PAD-UFES-20** as the external validation dataset

Since the two datasets do not contain identical diagnostic classes, this notebook performs label harmonization.

## Final Harmonized Classes

| ISIC2019 | PAD-UFES-20 | Final Label |
|----------|-------------|-------------|
| AK | ACK | AK |
| BCC | BCC | BCC |
| BKL | SEK | BKL |
| MEL | MEL | MEL |
| NV | NEV | NV |
| SCC | SCC | SCC |

Classes **DF** and **VASC** from ISIC2019 are excluded because PAD-UFES-20 does not contain equivalent categories.

This preprocessing decision ensures a fair cross-dataset evaluation.

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print("Libraries imported successfully.")

In [ ]:
ISIC_ROOT = Path("/kaggle/input/datasets/salviohexia/isic-2019-skin-lesion-images-for-classification")
PAD_ROOT = Path("/kaggle/input/datasets/mahdavi1202/skin-cancer")

PAD_METADATA_PATH = PAD_ROOT / "metadata.csv"

WORKING_DIR = Path("/kaggle/working")

PROCESSED_DIR = WORKING_DIR / "processed"
SPLITS_DIR = WORKING_DIR / "splits"
FIGURES_DIR = WORKING_DIR / "figures"
RESULTS_DIR = WORKING_DIR / "results"

for directory in [PROCESSED_DIR, SPLITS_DIR, FIGURES_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("ISIC Root:", ISIC_ROOT)
print("PAD Root:", PAD_ROOT)
print("PAD Metadata:", PAD_METADATA_PATH)

## 1. Load ISIC2019 Dataset

The ISIC2019 Kaggle dataset is organized by class folders.  
Each folder name is treated as the diagnostic label.

In [ ]:
ISIC_CLASSES = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]

isic_records = []

for label in ISIC_CLASSES:
    class_dir = ISIC_ROOT / label

    if not class_dir.exists():
        print("Missing folder:", class_dir)
        continue

    image_files = (
        list(class_dir.glob("*.jpg")) +
        list(class_dir.glob("*.jpeg")) +
        list(class_dir.glob("*.png"))
    )

    for img_path in image_files:
        isic_records.append({
            "dataset": "ISIC2019",
            "image_id": img_path.stem,
            "filename": img_path.name,
            "path": str(img_path),
            "original_label": label
        })

isic_raw_df = pd.DataFrame(isic_records)

print("ISIC raw shape:", isic_raw_df.shape)
display(isic_raw_df.head())

## 2. Load PAD-UFES-20 Dataset

PAD-UFES-20 images are stored in three folders.  
The metadata file contains the diagnostic label and image filename.

In [ ]:
pad_metadata = pd.read_csv(PAD_METADATA_PATH)

print("PAD metadata shape:", pad_metadata.shape)
print("PAD metadata columns:")
print(pad_metadata.columns.tolist())

display(pad_metadata.head())

In [ ]:
pad_image_files = []

for part in ["imgs_part_1", "imgs_part_2", "imgs_part_3"]:
    part_dir = PAD_ROOT / part / part

    if part_dir.exists():
        files = (
            list(part_dir.glob("*.png")) +
            list(part_dir.glob("*.jpg")) +
            list(part_dir.glob("*.jpeg"))
        )
        pad_image_files.extend(files)
        print(f"{part}: {len(files)} images")
    else:
        print("Missing folder:", part_dir)

print("Total PAD images:", len(pad_image_files))

In [ ]:
possible_image_columns = ["img_id", "image_id", "image", "filename", "file", "name"]
possible_label_columns = ["diagnostic", "diagnosis", "label", "class", "lesion_type"]

pad_image_col = None
pad_label_col = None

for col in possible_image_columns:
    if col in pad_metadata.columns:
        pad_image_col = col
        break

for col in possible_label_columns:
    if col in pad_metadata.columns:
        pad_label_col = col
        break

print("Detected PAD image column:", pad_image_col)
print("Detected PAD label column:", pad_label_col)

if pad_image_col is None or pad_label_col is None:
    raise ValueError("Could not detect PAD image or label column. Please inspect metadata columns.")

In [ ]:
pad_path_map = {}

for path in pad_image_files:
    pad_path_map[path.name] = str(path)
    pad_path_map[path.stem] = str(path)

pad_records = []

for _, row in pad_metadata.iterrows():
    image_value = str(row[pad_image_col])
    image_stem = Path(image_value).stem

    matched_path = None

    if image_value in pad_path_map:
        matched_path = pad_path_map[image_value]
    elif image_stem in pad_path_map:
        matched_path = pad_path_map[image_stem]

    pad_records.append({
        "dataset": "PADUFES20",
        "image_id": image_stem,
        "filename": image_value,
        "path": matched_path,
        "original_label": str(row[pad_label_col]).upper()
    })

pad_raw_df = pd.DataFrame(pad_records)

print("PAD raw shape:", pad_raw_df.shape)
display(pad_raw_df.head())

## 3. Label Harmonization

To perform cross-dataset validation, both datasets must use the same diagnostic label space.

The final six classes are:

- AK
- BCC
- BKL
- MEL
- NV
- SCC

ISIC labels `DF` and `VASC` are removed because they do not have equivalent labels in PAD-UFES-20.

In [ ]:
ISIC_LABEL_MAP = {
    "AK": "AK",
    "BCC": "BCC",
    "BKL": "BKL",
    "MEL": "MEL",
    "NV": "NV",
    "SCC": "SCC"
}

PAD_LABEL_MAP = {
    "ACK": "AK",
    "BCC": "BCC",
    "SEK": "BKL",
    "MEL": "MEL",
    "NEV": "NV",
    "SCC": "SCC"
}

FINAL_CLASSES = ["AK", "BCC", "BKL", "MEL", "NV", "SCC"]

print("Final classes:", FINAL_CLASSES)

In [ ]:
isic_df = isic_raw_df.copy()
pad_df = pad_raw_df.copy()

isic_df["label"] = isic_df["original_label"].map(ISIC_LABEL_MAP)
pad_df["label"] = pad_df["original_label"].map(PAD_LABEL_MAP)

isic_removed = isic_df[isic_df["label"].isnull()]
pad_removed = pad_df[pad_df["label"].isnull()]

print("ISIC samples before harmonization:", len(isic_raw_df))
print("ISIC samples removed:", len(isic_removed))
print("ISIC samples after harmonization:", len(isic_df.dropna(subset=["label"])))

print("\nPAD samples before harmonization:", len(pad_raw_df))
print("PAD samples removed:", len(pad_removed))
print("PAD samples after harmonization:", len(pad_df.dropna(subset=["label"])))

isic_df = isic_df.dropna(subset=["label"]).reset_index(drop=True)
pad_df = pad_df.dropna(subset=["label"]).reset_index(drop=True)

display(isic_df.head())
display(pad_df.head())

## 4. Remove Invalid Image Paths

Samples without valid image paths are removed from the processed dataset.

In [ ]:
before_isic = len(isic_df)
before_pad = len(pad_df)

isic_df = isic_df.dropna(subset=["path"]).reset_index(drop=True)
pad_df = pad_df.dropna(subset=["path"]).reset_index(drop=True)

isic_df = isic_df[isic_df["path"].apply(lambda x: Path(x).exists())].reset_index(drop=True)
pad_df = pad_df[pad_df["path"].apply(lambda x: Path(x).exists())].reset_index(drop=True)

print("ISIC removed missing paths:", before_isic - len(isic_df))
print("PAD removed missing paths:", before_pad - len(pad_df))

print("Final ISIC samples:", len(isic_df))
print("Final PAD samples:", len(pad_df))

## 5. Verify Image Readability

A small but important preprocessing step is verifying that images can be opened correctly.

In [ ]:
def verify_readable_images(df, dataset_name):
    valid_records = []
    corrupted_records = []

    for _, row in df.iterrows():
        try:
            img = Image.open(row["path"]).convert("RGB")
            img.verify()
            valid_records.append(row)
        except Exception as e:
            corrupted_records.append({
                "image_id": row["image_id"],
                "path": row["path"],
                "error": str(e)
            })

    valid_df = pd.DataFrame(valid_records).reset_index(drop=True)
    corrupted_df = pd.DataFrame(corrupted_records)

    print(f"{dataset_name} valid images:", len(valid_df))
    print(f"{dataset_name} corrupted images:", len(corrupted_df))

    return valid_df, corrupted_df


isic_df, isic_corrupted = verify_readable_images(isic_df, "ISIC2019")
pad_df, pad_corrupted = verify_readable_images(pad_df, "PADUFES20")

isic_corrupted.to_csv(RESULTS_DIR / "isic2019_corrupted_after_preprocessing.csv", index=False)
pad_corrupted.to_csv(RESULTS_DIR / "padufes20_corrupted_after_preprocessing.csv", index=False)

## 6. Class Distribution After Harmonization

This section verifies the final class distribution after label harmonization.

In [ ]:
def class_distribution(df, dataset_name):
    dist = df["label"].value_counts().reset_index()
    dist.columns = ["Class", "Count"]
    dist["Percentage"] = (dist["Count"] / dist["Count"].sum() * 100).round(2)
    dist["Dataset"] = dataset_name
    return dist[["Dataset", "Class", "Count", "Percentage"]]


isic_dist = class_distribution(isic_df, "ISIC2019")
pad_dist = class_distribution(pad_df, "PADUFES20")

display(isic_dist)
display(pad_dist)

isic_dist.to_csv(RESULTS_DIR / "isic2019_harmonized_class_distribution.csv", index=False)
pad_dist.to_csv(RESULTS_DIR / "padufes20_harmonized_class_distribution.csv", index=False)

In [ ]:
def plot_distribution(dist_df, title, filename):
    plt.figure(figsize=(10, 6))
    plt.bar(dist_df["Class"], dist_df["Count"])
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.tight_layout()

    save_path = FIGURES_DIR / filename
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)


plot_distribution(
    isic_dist,
    "ISIC2019 Class Distribution After Harmonization",
    "figure_09_isic2019_harmonized_distribution.png"
)

plot_distribution(
    pad_dist,
    "PAD-UFES-20 Class Distribution After Harmonization",
    "figure_10_padufes20_harmonized_distribution.png"
)

## 7. Encode Labels

The final class labels are converted into numeric labels for model training.

In [ ]:
label_to_index = {label: idx for idx, label in enumerate(FINAL_CLASSES)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

isic_df["label_index"] = isic_df["label"].map(label_to_index)
pad_df["label_index"] = pad_df["label"].map(label_to_index)

print("Label to index mapping:")
print(label_to_index)

with open(RESULTS_DIR / "label_mapping.json", "w") as f:
    json.dump({
        "label_to_index": label_to_index,
        "index_to_label": index_to_label
    }, f, indent=4)

display(isic_df.head())
display(pad_df.head())

## 8. Train / Validation / Test Split

ISIC2019 is used for training, validation, and internal testing.

PAD-UFES-20 is kept untouched as an external validation dataset.

We use a stratified split to preserve class distribution.

In [ ]:
train_df, temp_df = train_test_split(
    isic_df,
    test_size=0.30,
    stratify=isic_df["label"],
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
external_df = pad_df.reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Internal test samples:", len(test_df))
print("External PAD test samples:", len(external_df))

In [ ]:
split_summary = pd.concat([
    class_distribution(train_df, "Train"),
    class_distribution(val_df, "Validation"),
    class_distribution(test_df, "Internal Test"),
    class_distribution(external_df, "External PAD Test")
])

display(split_summary)

split_summary.to_csv(RESULTS_DIR / "split_class_distribution.csv", index=False)

## 9. Compute Class Weights

Because the dataset is imbalanced, class weights are computed from the ISIC training split.

These class weights will be used during model training.

In [ ]:
classes = np.array(sorted(train_df["label_index"].unique()))

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label_index"].values
)

class_weights = {int(cls): float(weight) for cls, weight in zip(classes, weights)}

print("Class weights:")
print(class_weights)

with open(RESULTS_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=4)

## 10. Save Processed CSV Files

The processed CSV files will be used by all later notebooks.

These files contain:

- image path
- image ID
- original label
- harmonized label
- numeric label index
- dataset split

In [ ]:
train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "internal_test"
external_df["split"] = "external_test"

train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "validation.csv", index=False)
test_df.to_csv(SPLITS_DIR / "internal_test.csv", index=False)
external_df.to_csv(SPLITS_DIR / "external_padufes20_test.csv", index=False)

isic_df.to_csv(PROCESSED_DIR / "isic2019_harmonized.csv", index=False)
pad_df.to_csv(PROCESSED_DIR / "padufes20_harmonized.csv", index=False)

all_splits_df = pd.concat([train_df, val_df, test_df, external_df]).reset_index(drop=True)
all_splits_df.to_csv(PROCESSED_DIR / "all_processed_splits.csv", index=False)

print("Saved processed CSV files.")
print("Splits directory:", SPLITS_DIR)
print("Processed directory:", PROCESSED_DIR)

## 11. Preprocessing Configuration

The final image input size for lightweight CNN models will be:

**224 × 224 × 3**

This size is compatible with MobileNetV2, MobileNetV3, EfficientNet, and TensorFlow Lite deployment.

In [ ]:
preprocessing_config = {
    "image_size": [224, 224],
    "channels": 3,
    "normalization": "model_specific_preprocessing",
    "final_classes": FINAL_CLASSES,
    "num_classes": len(FINAL_CLASSES),
    "isic_removed_classes": ["DF", "VASC"],
    "pad_label_mapping": PAD_LABEL_MAP,
    "isic_label_mapping": ISIC_LABEL_MAP,
    "train_split": len(train_df),
    "validation_split": len(val_df),
    "internal_test_split": len(test_df),
    "external_test_split": len(external_df),
    "random_state": RANDOM_STATE
}

with open(RESULTS_DIR / "preprocessing_config.json", "w") as f:
    json.dump(preprocessing_config, f, indent=4)

print("Preprocessing configuration saved.")

## 12. Sample Processed Images

This section displays random samples from the training data after label harmonization.

In [ ]:
def show_samples(df, title, samples_per_class=4):
    labels = FINAL_CLASSES

    for label in labels:
        class_df = df[df["label"] == label]

        if len(class_df) == 0:
            continue

        sample_df = class_df.sample(
            min(samples_per_class, len(class_df)),
            random_state=RANDOM_STATE
        )

        plt.figure(figsize=(14, 4))

        for i, (_, row) in enumerate(sample_df.iterrows()):
            img = Image.open(row["path"]).convert("RGB")

            plt.subplot(1, samples_per_class, i + 1)
            plt.imshow(img)
            plt.title(f"{label}")
            plt.axis("off")

        plt.suptitle(f"{title}: {label}")
        plt.tight_layout()

        save_path = FIGURES_DIR / f"processed_samples_{title.lower().replace(' ', '_')}_{label}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()


show_samples(train_df, "ISIC Train", samples_per_class=4)

## 13. Final Preprocessing Summary

This notebook generated a clean 6-class dataset setup suitable for cross-dataset validation.

The output of this notebook will be used directly in:

- `03_baseline_models.ipynb`
- `04_proposed_model.ipynb`
- `05_cross_dataset_validation.ipynb`
- `06_gradcam_xai.ipynb`
- `07_tflite_conversion.ipynb`

In [ ]:
print("=" * 70)
print("MobileSkinXAI Preprocessing Pipeline Completed")
print("=" * 70)

print("\nFinal Classes:")
print(FINAL_CLASSES)

print("\nISIC2019")
print("Total harmonized samples:", len(isic_df))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Internal test:", len(test_df))

print("\nPAD-UFES-20")
print("External test samples:", len(external_df))

print("\nRemoved ISIC classes:")
print(["DF", "VASC"])

print("\nSaved Outputs:")
print("Processed CSV:", PROCESSED_DIR)
print("Split CSV:", SPLITS_DIR)
print("Figures:", FIGURES_DIR)
print("Results:", RESULTS_DIR)